In [8]:
# ============================================================
# Cell 1 — Imports & Config
# ============================================================
import pandas as pd
import numpy as np
import time
from pathlib import Path

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_recall_curve, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)

RANDOM = 42
DATASET_PATH = Path("../dataset/bank-additional-full.csv")

# Column definitions (duration is dropped — it leaks the target)
NUMERIC_FEATURES = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx",
    "euribor3m", "nr.employed", "pdays_never_contacted"
]

CATEGORICAL_FEATURES = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month",
    "day_of_week", "poutcome"
]

TARGET = "y"

In [9]:
# ============================================================
# Cell 2 — Load data, drop duration, flag pdays, encode target
# ============================================================
df = pd.read_csv(DATASET_PATH, sep=";")

print(f"Full dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"Positive class: {df[TARGET].value_counts(normalize=True)['yes']:.2%}")

# Encode target
y = (df[TARGET] == "yes").astype(int)
df.drop(columns=[TARGET], inplace=True)

# Drop duration — recorded after the call ends, leaks the target
df.drop(columns=["duration"], inplace=True)

# Flag pdays==999 as a sentinel — "never previously contacted"
df["pdays_never_contacted"] = (df["pdays"] == 999).astype(int)

# Update NUMERIC_FEATURES to include the new sentinel column
NUMERIC_FEATURES = ["pdays_never_contacted"] + [
    f for f in NUMERIC_FEATURES if f != "pdays_never_contacted"
]

# 'unknown' values are kept as-is — they are informative categories, not missing data
print(f"\nAfter cleaning — features: {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")
print(f"  Numeric: {len(NUMERIC_FEATURES)} — {NUMERIC_FEATURES}")
print(f"  Categorical: {len(CATEGORICAL_FEATURES)} — {CATEGORICAL_FEATURES}")
print(f"  Target: {TARGET} (1=yes, 0=no)")

Full dataset shape: (41188, 21)

Target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64
Positive class: 11.27%

After cleaning — features: 20
  Numeric: 10 — ['pdays_never_contacted', 'age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
  Categorical: 10 — ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
  Target: y (1=yes, 0=no)


In [10]:
# ============================================================
# Cell 3 — Stratified 60/20/20 split (no leakage)
# ============================================================
# First: hold-out 20% test (unseen until final eval)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM
)

# Second: split trainval → train (60% of full) + val (20% of full)
# val_size = 0.25 of trainval = 0.25 × 0.80 = 0.20 of full
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    stratify=y_trainval,
    random_state=RANDOM
)

print("Split sizes:")
print(f"  Train : {len(X_train):,}  ({len(X_train)/len(df):.0%})")
print(f"  Val   : {len(X_val):,}  ({len(X_val)/len(df):.0%})")
print(f"  Test  : {len(X_test):,}  ({len(X_test)/len(df):.0%})")
print(f"\nClass balance check — Train: {y_train.mean():.2%}  Val: {y_val.mean():.2%}  Test: {y_test.mean():.2%}")

Split sizes:
  Train : 24,712  (60%)
  Val   : 8,238  (20%)
  Test  : 8,238  (20%)

Class balance check — Train: 11.27%  Val: 11.26%  Test: 11.26%


In [11]:
# ============================================================
# Cell 4 — Preprocessor (fit ONLY on train, transform val & test)
# ============================================================
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False),
     CATEGORICAL_FEATURES),
])

# Fit ONLY on train — never on val or test
X_train_tr = preprocessor.fit_transform(X_train)
X_val_tr = preprocessor.transform(X_val)
X_test_tr = preprocessor.transform(X_test)

# Get feature names after one-hot encoding
cat_names = preprocessor.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = list(NUMERIC_FEATURES) + list(cat_names)

print(f"Training matrix shape:    {X_train_tr.shape}")
print(f"Validation matrix shape:  {X_val_tr.shape}")
print(f"Test matrix shape:        {X_test_tr.shape}")
print(f"Total features after encoding: {len(all_feature_names)}")

Training matrix shape:    (24712, 63)
Validation matrix shape:  (8238, 63)
Test matrix shape:        (8238, 63)
Total features after encoding: 63


In [12]:
# ====================================================================
# Cell 5 — 5-fold CV on all 3 models | Per-fold threshold | Metrics
# ====================================================================
from sklearn.model_selection import StratifiedKFold
from collections import defaultdict

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=RANDOM
    ),
    "RandomForest": RandomForestClassifier(
        class_weight="balanced", n_estimators=200,
        max_depth=None, random_state=RANDOM, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced", max_iter=200,
        max_depth=None, random_state=RANDOM
    ),
}

CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM)

# Collect per-fold metrics for each model
cv_results = defaultdict(lambda: {
    "accuracy": [], "precision": [], "recall": [],
    "f1": [], "roc_auc": [], "threshold": [], "train_time": []
})

for name, est in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    for fold, (tr_idx, v_idx) in enumerate(skf.split(X_trainval, y_trainval)):
        X_tr_fold = X_trainval.iloc[tr_idx]
        y_tr_fold = y_trainval.iloc[tr_idx]
        X_v_fold = X_trainval.iloc[v_idx]
        y_v_fold = y_trainval.iloc[v_idx]

        # Fresh preprocessor per fold — fit ONLY on fold-train
        pp = ColumnTransformer([
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False),
             CATEGORICAL_FEATURES),
        ])
        X_tr_pp = pp.fit_transform(X_tr_fold)
        X_v_pp = pp.transform(X_v_fold)

        # Time & fit
        t0 = time.time()
        clone_est = sklearn.base.clone(est)
        clone_est.fit(X_tr_pp, y_tr_fold)
        train_time = time.time() - t0

        # Predict on fold-val
        y_v_proba = clone_est.predict_proba(X_v_pp)[:, 1]

        # Threshold: highest where recall >= 0.75
        precision_arr, recall_arr, thresholds = precision_recall_curve(y_v_fold, y_v_proba)
        valid_mask = recall_arr[:-1] >= 0.75
        if valid_mask.any():
            best_t = thresholds[valid_mask].max()
        else:
            best_t = 0.5

        y_v_pred = (y_v_proba >= best_t).astype(int)

        cv_results[name]["accuracy"].append(accuracy_score(y_v_fold, y_v_pred))
        cv_results[name]["precision"].append(precision_score(y_v_fold, y_v_pred))
        cv_results[name]["recall"].append(recall_score(y_v_fold, y_v_pred))
        cv_results[name]["f1"].append(f1_score(y_v_fold, y_v_pred))
        cv_results[name]["roc_auc"].append(roc_auc_score(y_v_fold, y_v_proba))
        cv_results[name]["threshold"].append(best_t)
        cv_results[name]["train_time"].append(train_time)

    fold_data = cv_results[name]
    print(f"  Accuracy  : {np.mean(fold_data['accuracy']):.4f} ± {np.std(fold_data['accuracy']):.4f}")
    print(f"  Precision : {np.mean(fold_data['precision']):.4f} ± {np.std(fold_data['precision']):.4f}")
    print(f"  Recall    : {np.mean(fold_data['recall']):.4f} ± {np.std(fold_data['recall']):.4f}")
    print(f"  F1        : {np.mean(fold_data['f1']):.4f} ± {np.std(fold_data['f1']):.4f}")
    print(f"  ROC-AUC   : {np.mean(fold_data['roc_auc']):.4f} ± {np.std(fold_data['roc_auc']):.4f}")
    print(f"  Threshold : {np.mean(fold_data['threshold']):.4f} ± {np.std(fold_data['threshold']):.4f}")
    print(f"  Train (s) : {np.mean(fold_data['train_time']):.1f}s avg")

print(f"\n{'='*60}")
print("  5-fold CV complete. Test set is still untouched.")
print(f"{'='*60}")


  LogisticRegression
  Accuracy  : 0.6714 ± 0.0210
  Precision : 0.2203 ± 0.0116
  Recall    : 0.7508 ± 0.0002
  F1        : 0.3404 ± 0.0139
  ROC-AUC   : 0.7896 ± 0.0054
  Threshold : 0.3621 ± 0.0063
  Train (s) : 1.9s avg

  RandomForest
  Accuracy  : 0.6373 ± 0.0260
  Precision : 0.2030 ± 0.0137
  Recall    : 0.7530 ± 0.0028
  F1        : 0.3195 ± 0.0166
  ROC-AUC   : 0.7707 ± 0.0082
  Threshold : 0.0580 ± 0.0062
  Train (s) : 0.8s avg

  HistGradientBoosting
  Accuracy  : 0.6782 ± 0.0097
  Precision : 0.2238 ± 0.0058
  Recall    : 0.7511 ± 0.0005
  F1        : 0.3448 ± 0.0068
  ROC-AUC   : 0.7956 ± 0.0027
  Threshold : 0.3493 ± 0.0036
  Train (s) : 0.2s avg

  5-fold CV complete. Test set is still untouched.


In [13]:
# ====================================================================
# Cell 6 — CV comparison table (mean ± std)
# ====================================================================
rows = []
for name in cv_results:
    d = cv_results[name]
    rows.append({
        "Model": name,
        "Accuracy": f"{np.mean(d['accuracy']):.4f} ± {np.std(d['accuracy']):.4f}",
        "Precision": f"{np.mean(d['precision']):.4f} ± {np.std(d['precision']):.4f}",
        "Recall": f"{np.mean(d['recall']):.4f} ± {np.std(d['recall']):.4f}",
        "F1": f"{np.mean(d['f1']):.4f} ± {np.std(d['f1']):.4f}",
        "ROC-AUC": f"{np.mean(d['roc_auc']):.4f} ± {np.std(d['roc_auc']):.4f}",
        "Threshold": f"{np.mean(d['threshold']):.4f} ± {np.std(d['threshold']):.4f}",
        "Train (s)": f"{np.mean(d['train_time']):.1f}",
    })

cv_df = pd.DataFrame(rows).set_index("Model")

# Highlight best mean columns
def highlight_best_cv(col_str):
    vals = []
    for v in col_str:
        try:
            vals.append(float(v.split()[0]))
        except ValueError:
            return ["" for _ in col_str]
    best = max(vals)
    return ["font-weight: bold; color: green" if float(v.split()[0]) == best else "" for v in col_str]

print("5-Fold Cross-Validation Results (trainval 80%) — mean ± std\n")
display(cv_df.style.apply(highlight_best_cv))
print("\n" + cv_df.to_string())
print("\nTest set still held out — waiting for final model selection.")

5-Fold Cross-Validation Results (trainval 80%) — mean ± std



,Accuracy,Precision,Recall,F1,ROC-AUC,Threshold,Train (s)
Model,,,,,,,
LogisticRegression,0.6714 ± 0.0210,0.2203 ± 0.0116,0.7508 ± 0.0002,0.3404 ± 0.0139,0.7896 ± 0.0054,0.3621 ± 0.0063,1.9
RandomForest,0.6373 ± 0.0260,0.2030 ± 0.0137,0.7530 ± 0.0028,0.3195 ± 0.0166,0.7707 ± 0.0082,0.0580 ± 0.0062,0.8
HistGradientBoosting,0.6782 ± 0.0097,0.2238 ± 0.0058,0.7511 ± 0.0005,0.3448 ± 0.0068,0.7956 ± 0.0027,0.3493 ± 0.0036,0.2



                             Accuracy        Precision           Recall               F1          ROC-AUC        Threshold Train (s)
Model                                                                                                                               
LogisticRegression    0.6714 ± 0.0210  0.2203 ± 0.0116  0.7508 ± 0.0002  0.3404 ± 0.0139  0.7896 ± 0.0054  0.3621 ± 0.0063       1.9
RandomForest          0.6373 ± 0.0260  0.2030 ± 0.0137  0.7530 ± 0.0028  0.3195 ± 0.0166  0.7707 ± 0.0082  0.0580 ± 0.0062       0.8
HistGradientBoosting  0.6782 ± 0.0097  0.2238 ± 0.0058  0.7511 ± 0.0005  0.3448 ± 0.0068  0.7956 ± 0.0027  0.3493 ± 0.0036       0.2

Test set still held out — waiting for final model selection.


In [14]:
# ====================================================================
# Cell 7 — Final pipeline, MLflow registration, model.joblib export
# ====================================================================
import hashlib
import json
import sys
from datetime import datetime, timezone
import mlflow
import joblib

WINNER = "HistGradientBoosting"
MODEL_NAME = "bank_marketing_pipeline"

# 1 ── Final pipeline (preprocessor + winning classifier) ────────
final_classifier = HistGradientBoostingClassifier(
    class_weight="balanced", max_iter=200, max_depth=None, random_state=RANDOM
)

final_pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False),
         CATEGORICAL_FEATURES),
    ])),
    ("classifier", final_classifier),
])

# 2 ── Retrain on trainval (80%) ──────────────────────────────────
final_pipeline.fit(X_trainval, y_trainval)
print("Pipeline fitted on trainval (80%).")

# 3 ── Operating threshold from 5-fold CV (already computed) ──────
thresh_folds = cv_results[WINNER]["threshold"]
operating_threshold = float(np.mean(thresh_folds))
print(f"Operating threshold (CV mean): {operating_threshold:.4f}")

# 4 ── Final blind evaluation on held-out test set ─────────────────
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= operating_threshold).astype(int)

test_acc  = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec  = recall_score(y_test, y_test_pred)
test_f1   = f1_score(y_test, y_test_pred)
test_auc  = roc_auc_score(y_test, y_test_proba)

print("\nFinal Test Set (20% held-out):")
print(f"  Accuracy :  {test_acc:.4f}")
print(f"  Precision:  {test_prec:.4f}")
print(f"  Recall   :  {test_rec:.4f}")
print(f"  F1       :  {test_f1:.4f}")
print(f"  ROC-AUC  :  {test_auc:.4f}")

# 5 ── MLflow artifact triple ──────────────────────────────────────
mlflow.set_experiment("bank_marketing_initial_training")

with mlflow.start_run(run_name=f"{WINNER}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}") as run:
    run_id = run.info.run_id

    # 5a — Binary artifact ────────────────────────────────────────
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )
    print(f"\nMLflow run: {run_id}")

    # 5b — Schema artifact ────────────────────────────────────────
    schema = {
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "target": {"name": TARGET, "classes": ["no", "yes"], "encoding": {"no": 0, "yes": 1}},
        "preprocessing": {
            "numeric": "StandardScaler",
            "categorical": "OneHotEncoder(handle_unknown='ignore', drop=None)",
        },
        "pipeline_steps": ["preprocessor", "classifier"],
        "classifier": "HistGradientBoostingClassifier",
        "hyperparameters": {
            "class_weight": "balanced",
            "max_iter": 200,
            "max_depth": None,
            "random_state": RANDOM,
        },
        "n_features_after_encoding": X_trainval.shape[1],
    }
    mlflow.log_dict(schema, "schema.json")

    # 5c — Model card artifact ────────────────────────────────────
    dataset_hash = hashlib.md5(DATASET_PATH.read_bytes()).hexdigest()

    import sklearn as _sk

    model_card = {
        "model_name": WINNER,
        "training_timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset": {
            "source": str(DATASET_PATH),
            "md5": dataset_hash,
            "rows": len(df),
            "features": len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES),
            "positive_class_ratio": round(y.mean(), 4),
        },
        "trainval_split": {
            "trainval_ratio": 0.80,
            "test_ratio": 0.20,
            "random_state": RANDOM,
            "stratified": True,
        },
        "cv_folds": CV_FOLDS,
        "operating_threshold": round(operating_threshold, 4),
        "threshold_rule": "highest threshold where recall >= 0.75 (per-fold, averaged)",
        "final_test_metrics": {
            "accuracy": round(test_acc, 4),
            "precision": round(test_prec, 4),
            "recall": round(test_rec, 4),
            "f1": round(test_f1, 4),
            "roc_auc": round(test_auc, 4),
        },
        "environment": {
            "python_version": sys.version.split()[0],
            "sklearn_version": _sk.__version__,
            "pandas_version": pd.__version__,
            "mlflow_version": mlflow.__version__,
        },
    }
    mlflow.log_dict(model_card, "model_card.json")

    print(f"Run ID       : {run_id}")
    print(f"Experiment   : {mlflow.get_experiment(mlflow.active_run().info.experiment_id).name}")
    print(f"Artifacts    : model/  schema.json  model_card.json")

# 6 ── Save model.joblib locally ──────────────────────────────────
joblib_path = Path("model.joblib")
joblib.dump(final_pipeline, joblib_path)
print(f"\nModel saved to {joblib_path.resolve()}")

print("\nDone.")

Pipeline fitted on trainval (80%).
Operating threshold (CV mean): 0.3493

Final Test Set (20% held-out):
  Accuracy :  0.6809
  Precision:  0.2301
  Recall   :  0.7812
  F1       :  0.3555
  ROC-AUC  :  0.8173


2026/05/05 15:09:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 15:09:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/05 15:09:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



MLflow run: bc1ee1e6d03c4218a0bd66c4d5d347a0
Run ID       : bc1ee1e6d03c4218a0bd66c4d5d347a0
Experiment   : bank_marketing_initial_training
Artifacts    : model/  schema.json  model_card.json

Model saved to /home/hadym/AIE-BOOTCAMP/Week5/project/initial-training/pipeline/model.joblib

Done.


Registered model 'bank_marketing_pipeline' already exists. Creating a new version of this model...
Created version '2' of model 'bank_marketing_pipeline'.
